# DAFTER spectral energy by frequency band

This notebook measures the spectrum produced by DAFTER's exact causal Mauer analysis and compares complex-magnitude normalizations

$$z_{normalized} = \beta |z|^{\alpha - 1} z,$$

so normalized magnitude is $\beta |z|^\alpha$. `alpha` changes the relative energy distribution; `beta` only changes the global scale. The Nyquist bin is removed, matching `DafterNetwork`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from after.autoencoder.audio import CausalMauerSTFT
from after.dataset import SimpleDataset

plt.style.use("seaborn-v0_8-whitegrid")

## Configuration

Set one or more LMDB paths below. Crops are hop-aligned and use the same 512-frame, 44.1 kHz representation as the training config. Add or edit normalization entries freely.

In [ ]:
DB_PATHS = ["/fast-1/nils/guitar_audio_midi"
    # Path("/path/to/dafter_guitar.lmdb"),
]

SAMPLE_RATE = 44_100
NFFT = 512
HOP_SIZE = 64
N_FRAMES = 512
MAX_EXAMPLES = 256
ANALYSIS_BATCH_SIZE = 16
SEED = 7
TARGET_PACKED_RMS = 1.0
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

NORMALISATIONS = [
    {"label": "current: alpha=.65 beta=.34", "alpha": 0.65, "beta": 0.34},
    {"label": "alpha=0.15 beta=.34", "alpha": 0.15, "beta": 0.34},
    {"label": "alpha=.25 beta=.34", "alpha": 0.25, "beta": 0.34},
    {"label": "alpha=0.5 beta=.34", "alpha": 0.5, "beta": 0.34},
    {"label": "alpha=.65 beta=1.0", "alpha": 0.65, "beta": 1.00},
]
GAUSSIAN_COMPARISON_LABEL = NORMALISATIONS[0]["label"]

BAND_EDGES_HZ = np.array([0, 125, 250, 500, 1_000, 2_000, 4_000,
                                8_000, 12_000, 16_000, SAMPLE_RATE / 2])

assert DB_PATHS, "Add at least one dataset path to DB_PATHS"
assert GAUSSIAN_COMPARISON_LABEL in {item["label"] for item in NORMALISATIONS}
print(f"device={DEVICE}, crop={N_FRAMES * HOP_SIZE / SAMPLE_RATE:.3f}s")

## Load deterministic random crops

Examples are sampled without replacement across all databases, proportional to database size. Stereo waveforms are averaged to mono, exactly as in `collate_dafter`.

In [ ]:
def mono_crop(waveform, crop_samples, hop_size, rng):
    waveform = np.asarray(waveform, dtype=np.float32)
    if waveform.ndim == 1:
        waveform = waveform[None]
    if waveform.shape[0] != 1:
        waveform = waveform.mean(axis=0, keepdims=True)
    if waveform.shape[-1] < crop_samples:
        waveform = np.pad(waveform, ((0, 0), (0, crop_samples - waveform.shape[-1])))
    available_hops = max(0, (waveform.shape[-1] - crop_samples) // hop_size)
    start_hop = 0 if available_hops == 0 else rng.integers(0, available_hops + 1)
    start = int(start_hop) * hop_size
    return waveform[..., start:start + crop_samples]


datasets = [SimpleDataset(path=str(Path(path)), keys=["waveform"]) for path in DB_PATHS]
lengths = np.array([len(dataset) for dataset in datasets], dtype=np.int64)
offsets = np.concatenate(([0], np.cumsum(lengths)))
total_examples = int(lengths.sum())
sample_count = min(MAX_EXAMPLES, total_examples)
rng = np.random.default_rng(SEED)
flat_indices = rng.choice(total_examples, size=sample_count, replace=False)
crop_samples = N_FRAMES * HOP_SIZE

crops = []
sampled_per_database = np.zeros(len(datasets), dtype=np.int64)
for flat_index in flat_indices:
    dataset_index = int(np.searchsorted(offsets[1:], flat_index, side="right"))
    item_index = int(flat_index - offsets[dataset_index])
    item = datasets[dataset_index][item_index]
    crops.append(mono_crop(item["waveform"], crop_samples, HOP_SIZE, rng))
    sampled_per_database[dataset_index] += 1

waveforms = torch.from_numpy(np.stack(crops)).float()
print(f"loaded {len(waveforms):,} crops with shape {tuple(waveforms.shape)}")
for path, size, sampled in zip(DB_PATHS, lengths, sampled_per_database):
    print(f"  {path}: {sampled:,} sampled from {size:,}")
print(f"waveform RMS={waveforms.square().mean().sqrt().item():.6f}, "
      f"peak={waveforms.abs().max().item():.6f}")

## Analyze raw spectra once

The expensive FFT is performed once. Every `(alpha, beta)` experiment is then evaluated from the same raw complex magnitudes. Energy means $|z_{normalized}|^2$, equivalent to real-squared plus imaginary-squared.

In [ ]:
analysis = CausalMauerSTFT(
    nfft=NFFT,
    hop_size=HOP_SIZE,
    synthesis_length=2 * HOP_SIZE,
    zero_length=HOP_SIZE,
    skip_features=-1,
    normalize=False,
    max_batch_size=1,
).to(DEVICE).eval()

frequency_hz = torch.fft.rfftfreq(NFFT, d=1 / SAMPLE_RATE)[:-1].numpy()
energy_sums = {setting["label"]: torch.zeros(NFFT // 2, dtype=torch.float64)
               for setting in NORMALISATIONS}
coefficient_sums = {setting["label"]: torch.zeros(2, NFFT // 2, dtype=torch.float64)
                    for setting in NORMALISATIONS}
coefficient_square_sums = {
    setting["label"]: torch.zeros(2, NFFT // 2, dtype=torch.float64)
    for setting in NORMALISATIONS}
magnitude_sum = torch.zeros(NFFT // 2, dtype=torch.float64)
magnitude_square_sum = torch.zeros(NFFT // 2, dtype=torch.float64)
coefficient_count = 0

with torch.no_grad():
    for start in range(0, len(waveforms), ANALYSIS_BATCH_SIZE):
        waveform_batch = waveforms[start:start + ANALYSIS_BATCH_SIZE].to(DEVICE)
        packed = analysis(waveform_batch)
        magnitude = torch.sqrt(packed[:, 0].square() + packed[:, 1].square()).clamp_min(1e-12)
        magnitude_sum += magnitude.sum(dim=(0, 2)).double().cpu()
        magnitude_square_sum += magnitude.square().sum(dim=(0, 2)).double().cpu()
        coefficient_count += magnitude.shape[0] * magnitude.shape[-1]
        for setting in NORMALISATIONS:
            label = setting["label"]
            complex_scale = setting["beta"] * magnitude.pow(setting["alpha"] - 1)
            normalized = packed * complex_scale[:, None]
            coefficient_sums[label] += normalized.sum(dim=(0, 3)).double().cpu()
            coefficient_square_sums[label] += normalized.square().sum(dim=(0, 3)).double().cpu()
            energy_sums[label] += normalized.square().sum(dim=(0, 1, 3)).double().cpu()

mean_energy = {label: values.numpy() / coefficient_count
               for label, values in energy_sums.items()}
coefficient_mean = {label: (values / coefficient_count).numpy()
                    for label, values in coefficient_sums.items()}
coefficient_std = {}
for label in coefficient_sums:
    second_moment = coefficient_square_sums[label] / coefficient_count
    variance = second_moment - (coefficient_sums[label] / coefficient_count).square()
    coefficient_std[label] = variance.clamp_min(0).sqrt().numpy()
mean_raw_magnitude = (magnitude_sum / coefficient_count).numpy()
rms_raw_magnitude = torch.sqrt(magnitude_square_sum / coefficient_count).numpy()
print(f"analyzed {coefficient_count:,} complex time-frequency coefficients per bin")

## Numerical summary

Packed RMS is the RMS over the two real-valued channels consumed by DAFTER, hence `sqrt(mean complex energy / 2)`. `beta for target RMS` shows the global multiplier required for the chosen `alpha`; it does not whiten individual frequency bins.

In [ ]:
summary_rows = []
for setting in NORMALISATIONS:
    label = setting["label"]
    energy = mean_energy[label]
    packed_rms = float(np.sqrt(energy.mean() / 2))
    beta_for_target = setting["beta"] * TARGET_PACKED_RMS / max(packed_rms, 1e-12)
    summary_rows.append((label, setting["alpha"], setting["beta"], packed_rms,
                         beta_for_target, 100 * energy[:32].sum() / energy.sum(),
                         100 * energy[128:].sum() / energy.sum()))

header = ("normalisation", "alpha", "beta", "packed RMS",
          f"beta for RMS={TARGET_PACKED_RMS:g}", "lowest 32 bins %", "upper half %")
widths = [max(len(header[i]), *(len(f"{row[i]:.6g}") if isinstance(row[i], float)
                              else len(str(row[i])) for row in summary_rows))
          for i in range(len(header))]
print(" | ".join(str(value).ljust(width) for value, width in zip(header, widths)))
print("-+-".join("-" * width for width in widths))
for row in summary_rows:
    formatted = [row[0]] + [f"{value:.6g}" for value in row[1:]]
    print(" | ".join(value.ljust(width) for value, width in zip(formatted, widths)))

## Real/imaginary moments versus white Gaussian noise

Unit white Gaussian noise has mean 0 and standard deviation 1 independently in every packed real and imaginary coordinate. The table pools all frequencies; the following plots retain frequency resolution. Whitening matches these first two moments but does not—and should not—remove musical correlations or make the target fully Gaussian.

In [ ]:
print(f"{'normalisation':38s} | {'real mean':>10s} | {'imag mean':>10s} | "
      f"{'real std':>10s} | {'imag std':>10s}")
print("-" * 94)
pooled_count = coefficient_count * (NFFT // 2)
for setting in NORMALISATIONS:
    label = setting["label"]
    pooled_mean = coefficient_sums[label].sum(dim=1) / pooled_count
    pooled_second = coefficient_square_sums[label].sum(dim=1) / pooled_count
    pooled_std = (pooled_second - pooled_mean.square()).clamp_min(0).sqrt()
    print(f"{label:38s} | {pooled_mean[0]:10.6f} | {pooled_mean[1]:10.6f} | "
          f"{pooled_std[0]:10.6f} | {pooled_std[1]:10.6f}")
print(f"{'unit white Gaussian':38s} | {0:10.6f} | {0:10.6f} | {1:10.6f} | {1:10.6f}")

In [ ]:
label = GAUSSIAN_COMPARISON_LABEL
means = coefficient_mean[label]
stds = coefficient_std[label]

fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)
axes[0].plot(frequency_hz, means[0], label="dataset real mean")
axes[0].plot(frequency_hz, means[1], label="dataset imaginary mean")
axes[0].axhline(0, color="black", linestyle="--", label="Gaussian mean = 0")
axes[0].set_ylabel("coefficient mean")
axes[0].set_title(f"Real/imaginary mean by frequency: {label}")
axes[0].legend()

axes[1].plot(frequency_hz, stds[0], label="dataset real std")
axes[1].plot(frequency_hz, stds[1], label="dataset imaginary std")
axes[1].axhline(1, color="black", linestyle="--", label="Gaussian std = 1")
axes[1].set_xlabel("frequency (Hz)")
axes[1].set_ylabel("coefficient standard deviation")
axes[1].set_title("Per-frequency scale compared with unit white Gaussian noise")
axes[1].legend()
for axis in axes:
    axis.set_xlim(0, SAMPLE_RATE / 2)
plt.tight_layout()
plt.show()

print("Note: the imaginary DC coefficient is structurally zero for real audio; "
      "it should remain zero rather than be whitened to unit variance.")

### Real/imaginary moments by frequency band

These are coefficient-level moments pooled over examples, frames, and bins inside each band. They are different from the later chart showing each band's percentage of total energy.

In [ ]:
label = GAUSSIAN_COMPARISON_LABEL
band_labels_for_moments = []
band_means = []
band_stds = []
for band_index, (low, high) in enumerate(zip(BAND_EDGES_HZ[:-1], BAND_EDGES_HZ[1:])):
    include_high = band_index == len(BAND_EDGES_HZ) - 2
    mask = ((frequency_hz >= low) &
            ((frequency_hz <= high) if include_high else (frequency_hz < high)))
    mask_tensor = torch.from_numpy(mask)
    count = coefficient_count * int(mask.sum())
    mean = coefficient_sums[label][:, mask_tensor].sum(dim=1) / count
    second = coefficient_square_sums[label][:, mask_tensor].sum(dim=1) / count
    std = (second - mean.square()).clamp_min(0).sqrt()
    band_labels_for_moments.append(f"{low:g}-{high:g}")
    band_means.append(mean.numpy())
    band_stds.append(std.numpy())
band_means = np.stack(band_means)
band_stds = np.stack(band_stds)

x = np.arange(len(band_labels_for_moments))
width = 0.38
fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)
axes[0].bar(x - width / 2, band_means[:, 0], width, label="real mean")
axes[0].bar(x + width / 2, band_means[:, 1], width, label="imaginary mean")
axes[0].axhline(0, color="black", linestyle="--", label="Gaussian mean = 0")
axes[0].set_ylabel("coefficient mean")
axes[0].set_title(f"Bandwise means: {label}")
axes[0].legend()
axes[1].bar(x - width / 2, band_stds[:, 0], width, label="real std")
axes[1].bar(x + width / 2, band_stds[:, 1], width, label="imaginary std")
axes[1].axhline(1, color="black", linestyle="--", label="Gaussian std = 1")
axes[1].set_ylabel("coefficient standard deviation")
axes[1].set_xlabel("frequency band (Hz)")
axes[1].set_title("Bandwise scale compared with unit white Gaussian noise")
axes[1].set_xticks(x, band_labels_for_moments, rotation=35, ha="right")
axes[1].legend()
plt.tight_layout()
plt.show()

## Per-frequency energy

The first plot shows absolute normalized energy. The second removes each curve's global scale to isolate changes in spectral shape. Curves with the same `alpha` but different `beta` should overlap in the relative plot.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 9), sharex=True)
for setting in NORMALISATIONS:
    label = setting["label"]
    energy = mean_energy[label]
    axes[0].plot(frequency_hz, 10 * np.log10(energy + 1e-20), label=label)
    axes[1].plot(frequency_hz, 10 * np.log10(energy / energy.max() + 1e-20), label=label)
axes[0].set_ylabel("mean energy (dB, arbitrary reference)")
axes[0].set_title("Absolute normalized complex-spectral energy")
axes[1].set_ylabel("energy relative to curve maximum (dB)")
axes[1].set_xlabel("frequency (Hz)")
axes[1].set_title("Spectral shape independent of global beta scale")
for axis in axes:
    axis.set_xlim(0, SAMPLE_RATE / 2)
    axis.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

## Energy by frequency band

In [ ]:
def frequency_label(value):
    return f"{value / 1000:g}k" if value >= 1000 else f"{value:g}"


band_labels = [f"{frequency_label(lo)}-{frequency_label(hi)}"
               for lo, hi in zip(BAND_EDGES_HZ[:-1], BAND_EDGES_HZ[1:])]
band_percentages = {}
for setting in NORMALISATIONS:
    energy = mean_energy[setting["label"]]
    values = []
    for band_index, (low, high) in enumerate(zip(BAND_EDGES_HZ[:-1], BAND_EDGES_HZ[1:])):
        include_high = band_index == len(BAND_EDGES_HZ) - 2
        mask = ((frequency_hz >= low) &
                ((frequency_hz <= high) if include_high else (frequency_hz < high)))
        values.append(100 * energy[mask].sum() / energy.sum())
    band_percentages[setting["label"]] = np.asarray(values)

x = np.arange(len(band_labels))
width = 0.82 / len(NORMALISATIONS)
fig, axis = plt.subplots(figsize=(14, 5.5))
for index, setting in enumerate(NORMALISATIONS):
    offset = (index - (len(NORMALISATIONS) - 1) / 2) * width
    axis.bar(x + offset, band_percentages[setting["label"]], width,
             label=setting["label"])
axis.set_xticks(x, band_labels, rotation=35, ha="right")
axis.set_ylabel("percentage of total normalized energy")
axis.set_xlabel("frequency band (Hz)")
axis.set_title("Energy distribution across frequency bands")
axis.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

## Cumulative energy

This makes it easy to read the frequency below which 50%, 90%, or 99% of normalized target energy lies.

In [ ]:
fig, axis = plt.subplots(figsize=(12, 5.5))
for setting in NORMALISATIONS:
    label = setting["label"]
    cumulative = np.cumsum(mean_energy[label]) / mean_energy[label].sum()
    axis.plot(frequency_hz, 100 * cumulative, label=label)
    quantiles = {}
    for quantile in (0.5, 0.9, 0.99):
        index = min(np.searchsorted(cumulative, quantile), len(frequency_hz) - 1)
        quantiles[f"f{int(100 * quantile)}"] = frequency_hz[index]
    print(label, ", ".join(f"{name}={value:.1f} Hz" for name, value in quantiles.items()))
for level in (50, 90, 99):
    axis.axhline(level, color="black", linewidth=0.6, alpha=0.35)
axis.set_xlim(0, SAMPLE_RATE / 2)
axis.set_ylim(0, 100.5)
axis.set_xlabel("frequency (Hz)")
axis.set_ylabel("cumulative energy (%)")
axis.set_title("Cumulative normalized spectral energy")
axis.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

## Raw-magnitude reference

These curves are independent of normalization and help identify whether high-frequency energy comes from the dataset itself or from aggressive magnitude compression.

In [ ]:
fig, axis = plt.subplots(figsize=(12, 5))
axis.plot(frequency_hz, 20 * np.log10(mean_raw_magnitude + 1e-20),
          label="mean magnitude")
axis.plot(frequency_hz, 20 * np.log10(rms_raw_magnitude + 1e-20),
          label="RMS magnitude")
axis.set_xlim(0, SAMPLE_RATE / 2)
axis.set_xlabel("frequency (Hz)")
axis.set_ylabel("raw magnitude (dB, arbitrary reference)")
axis.set_title("Unnormalized causal Mauer spectrum")
axis.legend()
plt.tight_layout()
plt.show()